# SBA 7(a) Loan Risk — Modeling Export Validation

**Stage:** Initial candidate-feature assessment, before leakage investigation.

This notebook validates the PostgreSQL export, checks chronological evaluation periods, reviews missingness, and separates candidate predictors from target and audit-only columns.

**Run location:** Start Jupyter from the repository’s `notebooks/` directory. The source export must exist at `../data/exports/sba_model_export.csv`; regenerate it using the documented PostgreSQL views if missing.

**Historical workflow note:** The 15 predictors selected here were **initial candidates**, *not* the final feature set. Later leakage diagnostics led us to exclude `term_months_model` and `invalid_term_flag`; the frozen final Logistic Regression uses 13 predictors. The original 15-feature stage is retained to show the investigation accurately. Predictor availability at approval is not independently verified for every retained field.


In [ ]:

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)


In [ ]:

from pathlib import Path

# Load the PostgreSQL modeling export
export_path = Path("../data/exports/sba_model_export.csv")
if not export_path.is_file():
    raise FileNotFoundError(f"Missing model export: {export_path}. Generate it from PostgreSQL first.")

df = pd.read_csv(
    export_path,
    parse_dates=["approval_date"],
    dtype={
        "sba_lender_id": "string",
        "lender_key": "string"
    }
)

print("Dataset shape:", df.shape)

df.head()


Dataset shape: (478953, 25)


,loan_row_id,approval_date,approval_fy,modeling_period,gross_approval,guarantee_pct,term_months_model,invalid_term_flag,initial_interest_rate_model,invalid_interest_rate_flag,jobs_supported,business_age_group,naics_sector,project_state,processing_method,collateral_indicator,revolver_status,interest_rate_type,business_type,chargeoff_within_5yr,sba_lender_id,lender_key,lender_name,gross_chargeoff_amount,post_snapshot_event_flag
0,1,2009-10-01,2010,Train,288000.0,90.0,120.0,0,6.00,0,18.0,Unknown,Accommodation and Food Services,TX,Preferred Lenders Program,Y,N,V,CORPORATION,0,0072188,FDIC:18609,American Bank of Commerce,0.0,0
1,2,2009-10-01,2010,Train,1200000.0,90.0,300.0,0,4.75,0,10.0,5+ Years,"Professional, Scientific and Technical Services",FL,Preferred Lenders Program,N,N,V,PARTNERSHIP,0,0455644,FDIC:58665,Live Oak Banking Company,0.0,0
2,3,2009-10-01,2010,Train,120000.0,90.0,90.0,0,5.25,0,4.0,3–4 Years,Manufacturing,TX,Preferred Lenders Program,N,N,V,CORPORATION,0,0057328,FDIC:06560,The Huntington National Bank,0.0,0
3,4,2009-10-01,2010,Train,150000.0,50.0,60.0,0,5.25,0,48.0,Startup,Accommodation and Food Services,MI,SBA Express Program,Y,N,V,CORPORATION,0,0057328,FDIC:06560,The Huntington National Bank,0.0,0
4,6,2009-10-01,2010,Train,25000.0,50.0,84.0,0,6.50,0,4.0,5+ Years,Construction,NY,SBA Express Program,Y,Y,V,CORPORATION,0,0056102,FDIC:17534,KeyBank National Association,0.0,0


In [ ]:

# Validate five-year charge-off outcomes

print(df["chargeoff_within_5yr"].value_counts())

print(
    "Charge-off rate:",
    round(df["chargeoff_within_5yr"].mean() * 100, 4),
    "%"
)


chargeoff_within_5yr
0    460127
1     18826
Name: count, dtype: int64
Charge-off rate: 3.9307 %


In [ ]:

# Display all exported column names

print("Total columns:", len(df.columns))

for i, column in enumerate(df.columns, start=1):
    print(f"{i}. {column}")


Total columns: 25
1. loan_row_id
2. approval_date
3. approval_fy
4. modeling_period
5. gross_approval
6. guarantee_pct
7. term_months_model
8. invalid_term_flag
9. initial_interest_rate_model
10. invalid_interest_rate_flag
11. jobs_supported
12. business_age_group
13. naics_sector
14. project_state
15. processing_method
16. collateral_indicator
17. revolver_status
18. interest_rate_type
19. business_type
20. chargeoff_within_5yr
21. sba_lender_id
22. lender_key
23. lender_name
24. gross_chargeoff_amount
25. post_snapshot_event_flag


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 478953 entries, 0 to 478952
Data columns (total 25 columns):
 #   Column                       Non-Null Count   Dtype         
---  ------                       --------------   -----         
 0   loan_row_id                  478953 non-null  int64         
 1   approval_date                478953 non-null  datetime64[ns]
 2   approval_fy                  478953 non-null  int64         
 3   modeling_period              478953 non-null  object        
 4   gross_approval               478953 non-null  float64       
 5   guarantee_pct                478953 non-null  float64       
 6   term_months_model            478715 non-null  float64       
 7   invalid_term_flag            478953 non-null  int64         
 8   initial_interest_rate_model  478946 non-null  float64       
 9   invalid_interest_rate_flag   478953 non-null  int64         
 10  jobs_supported               478953 non-null  float64       
 11  business_age_group        

In [ ]:

# Calculate missing values by column

missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(4)
})

missing_summary = missing_summary.sort_values(
    "missing_count",
    ascending=False
)

display(missing_summary)


,missing_count,missing_pct
term_months_model,238,0.0497
initial_interest_rate_model,7,0.0015
loan_row_id,0,0.0000
processing_method,0,0.0000
gross_chargeoff_amount,0,0.0000
lender_name,0,0.0000
lender_key,0,0.0000
sba_lender_id,0,0.0000
chargeoff_within_5yr,0,0.0000
business_type,0,0.0000


In [ ]:

# Check whether every exported record has a unique ID

total_records = len(df)

unique_ids = df["loan_row_id"].nunique()

duplicate_ids = df["loan_row_id"].duplicated().sum()

print("Total records:", total_records)
print("Unique loan IDs:", unique_ids)
print("Duplicate loan IDs:", duplicate_ids)

assert total_records == 478953
assert unique_ids == total_records
assert duplicate_ids == 0

print("Record ID validation passed!")


Total records: 478953
Unique loan IDs: 478953
Duplicate loan IDs: 0
Record ID validation passed!


In [ ]:

# Summarize the chronological modeling periods

period_summary = (
    df.groupby("modeling_period")
      .agg(
          total_loans=("loan_row_id", "count"),
          chargeoffs=("chargeoff_within_5yr", "sum"),
          earliest_fy=("approval_fy", "min"),
          latest_fy=("approval_fy", "max")
      )
)

# Calculate the charge-off rate for each period
period_summary["chargeoff_rate_pct"] = (
    100 * period_summary["chargeoffs"]
    / period_summary["total_loans"]
).round(4)

# Display periods in chronological order
period_order = [
    "Train",
    "Tune",
    "Calibrate",
    "Validate",
    "Test"
]

period_summary = period_summary.reindex(period_order)

display(period_summary)

# Verify that no records are missing from the splits
assert period_summary["total_loans"].sum() == 478953
assert period_summary["chargeoffs"].sum() == 18826
assert period_summary["total_loans"].notna().all()

print("Chronological split validation passed!")


,total_loans,chargeoffs,earliest_fy,latest_fy,chargeoff_rate_pct
modeling_period,,,,,
Train,266226,9894,2010,2015,3.7164
Tune,56785,2198,2016,2016,3.8707
Calibrate,56078,2419,2017,2017,4.3136
Validate,54194,2529,2018,2018,4.6666
Test,45670,1786,2019,2019,3.9107


Chronological split validation passed!


In [ ]:

# Define approval-time candidate features

numeric_features = [
    "gross_approval",
    "guarantee_pct",
    "term_months_model",
    "initial_interest_rate_model",
    "jobs_supported"
]

categorical_features = [
    "business_age_group",
    "naics_sector",
    "project_state",
    "processing_method",
    "collateral_indicator",
    "revolver_status",
    "interest_rate_type",
    "business_type"
]

# Optional data-quality indicators
flag_features = [
    "invalid_term_flag",
    "invalid_interest_rate_flag"
]

feature_columns = (
    numeric_features
    + categorical_features
    + flag_features
)

target_column = "chargeoff_within_5yr"

print("Total candidate features:", len(feature_columns))
print("Target:", target_column)


Total candidate features: 15
Target: chargeoff_within_5yr


In [ ]:

# Fields that must never enter the predictor matrix

excluded_columns = [
    "loan_row_id",
    "approval_date",
    "approval_fy",
    "modeling_period",
    "chargeoff_within_5yr",
    "sba_lender_id",
    "lender_key",
    "lender_name",
    "gross_chargeoff_amount",
    "post_snapshot_event_flag"
]

# Check for missing feature names
missing_features = set(feature_columns) - set(df.columns)

# Check whether excluded fields were accidentally selected
leaked_features = set(feature_columns) & set(excluded_columns)

assert not missing_features, f"Missing features: {missing_features}"
assert not leaked_features, f"Excluded fields selected: {leaked_features}"
assert len(feature_columns) == len(set(feature_columns))

# Verify every exported column has an assigned role
assert set(feature_columns + excluded_columns) == set(df.columns)

print("Feature separation validation passed!")
print("Candidate predictors:", len(feature_columns))
print("Excluded target/audit fields:", len(excluded_columns))


Feature separation validation passed!
Candidate predictors: 15
Excluded target/audit fields: 10
